In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

D:\langchain_acedemy\lca-lc-foundations\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [3]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [12]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [5]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John, '
                                                                          'no '
                                                                          'problem, '
                                                                          'thanks '
                                                                          'for '
                                                                          'letting '
                                                                          'me '
                                                                          'know. '
                                                                          'What '
                                                                          'time '
                                                                          'works '
           

In [6]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'Hi John, no problem, thanks for letting me know. What time works best for you to reschedule? Best, Seán.'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': 'Hi John, no problem, thanks for letting me know. What time works best for you to reschedule? Best, Seán.'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='a2aeadc64d517f323305f77d4526afbd')]


In [13]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John, no problem, thanks for letting me know. What time works best for you to reschedule? Best, Seán.


## Approve

In [8]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='7678723d-16e9-4958-9abc-b57efa150bca'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{"address": "me"}'}, '__gemini_function_call_thought_signatures__': {'0037dc96-9d80-4bf7-85f7-3db4fb8df381': 'EjQKMgEMOdbHDxOP8B5KIle6TtbcjbHSkfmPl/aXPwz09HzfaMMXuJuj7EEh8gaWTHC/tmGj'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e54ea-6619-7113-bfaf-55f887fb688e-0', tool_calls=[{'name': 'read_email', 'args': {'address': 'me'}, 'id': '0037dc96-9d80-4bf7-85f7-3db4fb8df381', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_to

## Reject

In [14]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='7678723d-16e9-4958-9abc-b57efa150bca'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{"address": "me"}'}, '__gemini_function_call_thought_signatures__': {'0037dc96-9d80-4bf7-85f7-3db4fb8df381': 'EjQKMgEMOdbHDxOP8B5KIle6TtbcjbHSkfmPl/aXPwz09HzfaMMXuJuj7EEh8gaWTHC/tmGj'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e54ea-6619-7113-bfaf-55f887fb688e-0', tool_calls=[{'name': 'read_email', 'args': {'address': 'me'}, 'id': '0037dc96-9d80-4bf7-85f7-3db4fb8df381', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_to

In [15]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

KeyError: '__interrupt__'

## Edit

In [11]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': "Hi Seán, I'm going to be late for our meeting tomorrow. Can we "
          'reschedule? Best, John.',
 'messages': [HumanMessage(content='Please read my email and send a response immediately. Send the reply now in the same thread.', additional_kwargs={}, response_metadata={}, id='7678723d-16e9-4958-9abc-b57efa150bca'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'read_email', 'arguments': '{"address": "me"}'}, '__gemini_function_call_thought_signatures__': {'0037dc96-9d80-4bf7-85f7-3db4fb8df381': 'EjQKMgEMOdbHDxOP8B5KIle6TtbcjbHSkfmPl/aXPwz09HzfaMMXuJuj7EEh8gaWTHC/tmGj'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e54ea-6619-7113-bfaf-55f887fb688e-0', tool_calls=[{'name': 'read_email', 'args': {'address': 'me'}, 'id': '0037dc96-9d80-4bf7-85f7-3db4fb8df381', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_to